# BDD100K Multi-Label Scene Classification
**ResNet18 · BCEWithLogitsLoss · Grad-CAM**

Labels: `rain` · `night` · `clear` · `daytime`  
Dataset: Berkeley DeepDrive BDD100K (100k driving images)  
Training: 1,000 images · Validation: 300 images · 8 epochs · Apple MPS

---

## 0  Setup

In [ ]:
import sys, os
from pathlib import Path

# Add vision/ to path regardless of where the notebook is launched from
NOTEBOOK_DIR = Path(os.getcwd())
VISION_DIR   = NOTEBOOK_DIR.parent / 'vision'
sys.path.insert(0, str(VISION_DIR))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from PIL import Image
from torch.utils.data import DataLoader

from dataset import parse_bdd100k_json, BDD100KDataset, LABEL_NAMES, get_val_transforms
from model   import BDD100KClassifier
from gradcam import GradCAM, overlay_heatmap, denormalize
from metrics import MetricTracker, per_label_f1, exact_match_ratio, hamming_loss, binarize

print('Imports OK')
print('Label names:', LABEL_NAMES)

In [ ]:
# ── Paths (pre-filled to match this machine's BDD100K archive) ────────────────
ARCHIVE     = Path('/Users/sahojitkarmakar/Documents/ca traffic/traffic-system/archive (15)')

TRAIN_JSON  = ARCHIVE / 'bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json'
VAL_JSON    = ARCHIVE / 'bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json'
TRAIN_IMGS  = ARCHIVE / 'bdd100k/bdd100k/images/100k/train'
VAL_IMGS    = ARCHIVE / 'bdd100k/bdd100k/images/100k/val'
CHECKPOINT  = NOTEBOOK_DIR.parent / 'models/best_model.pth'

DEVICE = (
    torch.device('mps')  if torch.backends.mps.is_available() else
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('cpu')
)

print(f'Device     : {DEVICE}')
print(f'Train JSON : {TRAIN_JSON.exists()}')
print(f'Val JSON   : {VAL_JSON.exists()}')
print(f'Train imgs : {TRAIN_IMGS.exists()}')
print(f'Val imgs   : {VAL_IMGS.exists()}')
print(f'Checkpoint : {CHECKPOINT.exists()}')

---
## 1  Dataset Overview

In [ ]:
# Parse both annotation files
train_map = parse_bdd100k_json(str(TRAIN_JSON))
val_map   = parse_bdd100k_json(str(VAL_JSON))

print(f'Train annotations : {len(train_map):,}')
print(f'Val   annotations : {len(val_map):,}')

# Label distribution on train set (all 69k entries)
arr_all = np.stack(list(train_map.values()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart — positive rate
colors = ['#e74c3c', '#9b59b6', '#3498db', '#f39c12']
pos_rate = arr_all.mean(axis=0)
bars = axes[0].bar(LABEL_NAMES, pos_rate, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_ylabel('Positive rate', fontsize=11)
axes[0].set_title('Label distribution — full train set (69,863 images)', fontsize=11)
axes[0].set_ylim(0, 1)
for b, v in zip(bars, pos_rate):
    axes[0].text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')

# Co-occurrence heatmap
cooc = (arr_all.T @ arr_all) / len(arr_all)
im = axes[1].imshow(cooc, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks(range(4)); axes[1].set_yticks(range(4))
axes[1].set_xticklabels(LABEL_NAMES, rotation=30, ha='right')
axes[1].set_yticklabels(LABEL_NAMES)
axes[1].set_title('Label co-occurrence matrix', fontsize=11)
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f'{cooc[i,j]:.2f}', ha='center', va='center',
                     fontsize=9, color='white' if cooc[i,j] > 0.4 else 'black')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

---
## 2  Sample Images — one per label combination

In [ ]:
ds_raw = BDD100KDataset(
    image_dir=str(TRAIN_IMGS),
    json_path=str(TRAIN_JSON),
    transform=None,
    max_samples=1000,
)

seen, samples = set(), []
for i in range(len(ds_raw)):
    lbl = tuple(ds_raw.label_vectors[i].astype(int))
    if lbl not in seen:
        seen.add(lbl)
        img = Image.open(ds_raw.image_paths[i]).convert('RGB')
        samples.append((img, lbl, Path(ds_raw.image_paths[i]).name))
    if len(samples) >= 8:
        break

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.patch.set_facecolor('#0e1117')
for ax, (img, lbl, name) in zip(axes.flat, samples):
    ax.imshow(img)
    active = [LABEL_NAMES[i] for i, v in enumerate(lbl) if v] or ['(none)']
    ax.set_title(' + '.join(active), fontsize=9, color='white', pad=4)
    ax.set_xlabel(name[:20], fontsize=7, color='gray')
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')
plt.suptitle('BDD100K — sample image per label combination', color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print(f'Unique label combinations found: {len(seen)}')

---
## 3  Training Results Summary

Training ran for **8 epochs** on **1,000 train / 300 val** images using Apple MPS (M-series GPU).  
Best checkpoint saved at epoch 8 — **val macro F1 = 0.7812**.

In [ ]:
# Training curves (recorded from actual training run)
epochs      = list(range(1, 9))
train_loss  = [0.5499, 0.4776, 0.3965, 0.3009, 0.2489, 0.2132, 0.1944, 0.1652]
val_loss    = [0.5248, 0.4520, 0.2776, 0.2617, 0.2681, 0.2923, 0.2651, 0.2535]
train_f1    = [0.3972, 0.4298, 0.6255, 0.7622, 0.7953, 0.8500, 0.8688, 0.8971]
val_f1      = [0.4154, 0.5293, 0.7568, 0.7669, 0.7522, 0.7335, 0.7718, 0.7812]
val_exact   = [0.3300, 0.4067, 0.6467, 0.6833, 0.6733, 0.6133, 0.6467, 0.6967]
val_hamming = [0.2767, 0.2217, 0.1050, 0.0942, 0.0983, 0.1217, 0.1050, 0.0883]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(epochs, train_loss, 'o-', color='#4f7dff', label='Train')
axes[0].plot(epochs, val_loss,   's--', color='#f6ad55', label='Val')
axes[0].set_title('BCEWithLogitsLoss', fontsize=11)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Macro F1
axes[1].plot(epochs, train_f1, 'o-', color='#4f7dff', label='Train')
axes[1].plot(epochs, val_f1,   's--', color='#f6ad55', label='Val')
axes[1].axhline(0.7812, color='#48bb78', linestyle=':', linewidth=1.5, label='Best val (0.781)')
axes[1].set_title('Macro F1', fontsize=11)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Exact match & Hamming
axes[2].plot(epochs, val_exact,   '^-',  color='#68d391', label='Exact match')
axes[2].plot(epochs, val_hamming, 'v--', color='#fc8181', label='Hamming loss')
axes[2].set_title('Exact Match & Hamming Loss (val)', fontsize=11)
axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Training Curves — BDD100K ResNet18', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 4  Evaluate on Real Val Set

In [ ]:
model = BDD100KClassifier(pretrained=False)
ckpt  = torch.load(str(CHECKPOINT), map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval().to(DEVICE)
print(f"Checkpoint epoch : {ckpt['epoch']}")
print(f"Val macro F1     : {ckpt['val_metrics']['macro_f1']:.4f}")

val_ds = BDD100KDataset(
    image_dir=str(VAL_IMGS),
    json_path=str(VAL_JSON),
    transform=get_val_transforms(),
    max_samples=300,
)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)

tracker = MetricTracker()
with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images.to(DEVICE))
        tracker.update(logits, labels)

m = tracker.compute()

# Pretty table
print(f"\n{'Metric':<22} {'Score':>7}")
print('-' * 32)
print(f"{'Macro F1':<22} {m['macro_f1']:>7.4f}")
print(f"{'Exact Match':<22} {m['exact_match']:>7.4f}  ({m['exact_match']*300:.0f}/300 correct)")
print(f"{'Hamming Loss':<22} {m['hamming_loss']:>7.4f}")
print('-' * 32)
for name in LABEL_NAMES:
    print(f"  F1 [{name:<10}] {m[f'f1_{name}']:>7.4f}")

In [ ]:
# Per-label F1 bar chart
f1_scores = [m[f'f1_{n}'] for n in LABEL_NAMES]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(LABEL_NAMES, f1_scores, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(m['macro_f1'], color='white', linestyle='--', linewidth=1.2, label=f"Macro F1 = {m['macro_f1']:.3f}")
ax.set_ylim(0, 1.05)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('Per-label F1 on BDD100K val (300 images)', fontsize=11)
ax.legend()
for b, v in zip(bars, f1_scores):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5  Live Inference on Val Images

In [ ]:
transform = get_val_transforms()
sample_paths = val_ds.image_paths[:10]
gt_labels    = val_ds.label_vectors[:10]

results = []
for path, gt in zip(sample_paths, gt_labels):
    pil = Image.open(path).convert('RGB')
    t   = transform(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.sigmoid(model(t)).squeeze().cpu().tolist()
    pred = [int(p >= 0.5) for p in probs]
    results.append({'name': Path(path).name[:28], 'probs': probs, 'pred': pred, 'gt': gt.tolist()})

# Print table
header = f"{'Image':<30}  {'rain':>5}  {'night':>5}  {'clear':>5}  {'day':>5}  Correct?"
print(header)
print('-' * len(header))
for r in results:
    ps = '  '.join(f'{p:>5.2f}' for p in r['probs'])
    match = '✓' if r['pred'] == r['gt'] else '✗'
    print(f"{r['name']:<30}  {ps}  {match}")

---
## 6  Grad-CAM — What the model looks at

In [ ]:
grad_cam = GradCAM(model, DEVICE)

# Pick 3 diverse val images
pick_names = sorted(VAL_IMGS.glob('*.jpg'))[:3]

fig = plt.figure(figsize=(20, 4 * len(pick_names)))
gs  = gridspec.GridSpec(len(pick_names), len(LABEL_NAMES) + 1, figure=fig, hspace=0.4, wspace=0.05)

for row, img_path in enumerate(pick_names):
    pil    = Image.open(img_path).convert('RGB')
    tensor = transform(pil).unsqueeze(0)
    orig   = denormalize(tensor)

    with torch.no_grad():
        probs = torch.sigmoid(model(tensor.to(DEVICE))).squeeze().cpu().tolist()

    # Original
    ax = fig.add_subplot(gs[row, 0])
    ax.imshow(orig); ax.axis('off')
    ax.set_title(img_path.name[:18] + '\n' + ' | '.join(f'{n}:{p:.2f}' for n,p in zip(LABEL_NAMES,probs)),
                 fontsize=7)

    # One heatmap per label
    for col, (label, prob) in enumerate(zip(LABEL_NAMES, probs), start=1):
        heatmap = grad_cam.generate(tensor.clone(), label_idx=col - 1)
        overlay = overlay_heatmap(orig.copy(), heatmap, alpha=0.45)
        ax = fig.add_subplot(gs[row, col])
        ax.imshow(overlay); ax.axis('off')
        tick = '✓' if prob >= 0.5 else '✗'
        ax.set_title(f'{label}\np={prob:.2f} {tick}', fontsize=8,
                     color='#4ade80' if prob >= 0.5 else '#aaa')

# Column headers
for col, title in enumerate(['Original'] + LABEL_NAMES):
    fig.axes[col].set_title(title, fontsize=9, fontweight='bold')

plt.suptitle('Grad-CAM heatmaps — layer4 activations per label', fontsize=13, y=1.01)
plt.savefig('../outputs/gradcam_notebook.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → outputs/gradcam_notebook.png')

---
## 7  Prediction Confidence Distribution

In [ ]:
all_probs = []
all_gt    = []

with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images.to(DEVICE))
        all_probs.append(torch.sigmoid(logits).cpu())
        all_gt.append(labels)

all_probs = torch.cat(all_probs).numpy()   # (N, 4)
all_gt    = torch.cat(all_gt).numpy()      # (N, 4)

fig, axes = plt.subplots(1, 4, figsize=(16, 3), sharey=True)
for i, (name, ax) in enumerate(zip(LABEL_NAMES, axes)):
    pos = all_probs[all_gt[:, i] == 1, i]
    neg = all_probs[all_gt[:, i] == 0, i]
    ax.hist(neg, bins=20, alpha=0.6, color='#fc8181', label=f'GT=0 (n={len(neg)})', density=True)
    ax.hist(pos, bins=20, alpha=0.6, color='#68d391', label=f'GT=1 (n={len(pos)})', density=True)
    ax.axvline(0.5, color='white', linestyle='--', linewidth=1)
    ax.set_title(f'{name}', fontsize=11)
    ax.set_xlabel('Predicted probability')
    ax.legend(fontsize=7)

axes[0].set_ylabel('Density')
plt.suptitle('Prediction confidence distribution per label (val set)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

| Metric | Value |
|--------|-------|
| **Val Macro F1** | **0.7812** |
| Exact Match | 69.7% (209/300) |
| Hamming Loss | 0.088 |
| F1 — rain | 0.400 |
| F1 — night | **0.983** |
| F1 — clear | **0.849** |
| F1 — daytime | **0.893** |

**Key observations:**
- `night` and `daytime` are learned most reliably — strong contrast in visual features
- `clear` vs `rain` is harder — rain is only 10.5% of the training subset (class imbalance)
- Grad-CAM shows the model attends to the sky/horizon for weather, headlights for night
- Adding `pos_weight` to `BCEWithLogitsLoss` for the `rain` label would likely improve F1 from 0.40 → 0.55+